# **融合算子概念介绍**
本章将通过模拟代码辅助讲解融合算子的基础理论，内容涵盖基本概念、核心优势与典型应用场景。我们将按照以下结构，带你逐步理解融合算子：
- 什么是融合算子
- CV融合算子介绍
- 融合算子的优势
- 课后练习
---

## **1. 什么是融合算子**
融合算子是将多个独立的基础小算子合并为一个整体算子，其功能等价于原多个算子组合执行的效果。开发者可根据实际业务场景与算法逻辑，灵活实现算子的自由融合，从而获得显著的性能优化收益。将Cube算子与Vector算子融合得到的算子称为CV融合算子，也称作Mix融合算子。


---
## **2. CV融合算子介绍**
对于存在数据依赖关系的矢量算子与矩阵算子，可通过融合算子编程将二者进行算子融合，统一由单个算子 Kernel 函数承载执行，从而获得显著的性能收益。下图对比了独立矢量算子、矩阵算子与CV融合算子的执行耗时，直观体现了开发CV融合算子的性能优势。
<img src="./images/02.07/mix_process.png" alt="turing_test"  width="700px">

接下来将以Matmul算子后接Add算子的典型场景为例，模拟通过Cube核与Vector核的流水线调度优化，直观展示算子融合技术对硬件算力利用率的提升效果。

一、融合前（串行执行）
所有数据先完成全量 Matmul（Cube 核）运算，再统一执行全量 Add（Vector 核）运算，  
单块执行流程为：数据块Cube搬入 → 数据块Cube运算 → 数据块Cube搬出待 4 块数据全部完成 Cube 核流程后，再逐块执行：数据块Vector搬入 → 数据块Vector运算 → 数据块Vector搬出  

<img src="./images/02.07/serial_timeline_before_fusion.png" alt="pipeline_timeline_before_fusion" width="700px" />  

二、融合后（流水线并行执行）
将数据切分为 4 块，通过流水线调度实现 Cube 核与 Vector 核并行工作，具体执行时序：
1. 数据 1：Cube 搬入 → Cube 运算 → Cube 搬出
2. 数据 1Vector 搬入 同时 数据 2Cube 搬入
3. 数据 1Vector 运算 同时 数据 2Cube 运算
4. 数据 1Vector 搬出 同时 数据 2Cube 搬出
5. 数据 2Vector 搬入 同时 数据 3Cube 搬入
6. 数据 2Vector 运算 同时 数据 3Cube 运算
7. 数据 2Vector 搬出 同时 数据 3Cube 搬出
8. 数据 3Vector 搬入 同时 数据 4Cube 搬入
9. 数据 3Vector 运算 同时 数据 4Cube 运算
10. 数据 3Vector 搬出 同时 数据 4Cube 搬出
11. 数据 4Vector 搬入 → 数据 4Vector 运算 → 数据 4Vector 搬出  

<img src="./images/02.07/pipeline_timeline_after_fusion.png" alt="pipeline_timeline_after_fusion" width="700px" />  


**核心优势：**  
融合前 Cube 核与 Vector 核交替闲置，总耗时为全量 Cube 流程 + 全量 Vector 流程；融合后通过流水线并行，前一块数据的 Vector 运算与后一块数据的 Cube 运算同步执行，最大化利用硬件算力，显著降低总耗时。

In [ ]:
import numpy as np

# ===================== 单步耗时定义（单位：秒） =====================
CUBE_MOVE_IN = 0.1     # Cube核搬入耗时
CUBE_COMPUTE = 0.2     # Cube核运算耗时
CUBE_MOVE_OUT = 0.1    # Cube核搬出耗时
VECTOR_MOVE_IN = 0.05  # Vector核搬入耗时
VECTOR_COMPUTE = 0.1   # Vector核运算耗时
VECTOR_MOVE_OUT = 0.05 # Vector核搬出耗时

# 单块Cube全流程耗时（搬入+运算+搬出）
SINGLE_CUBE = CUBE_MOVE_IN + CUBE_COMPUTE + CUBE_MOVE_OUT  # 0.4s
# 单块Vector全流程耗时（搬入+运算+搬出）
SINGLE_VECTOR = VECTOR_MOVE_IN + VECTOR_COMPUTE + VECTOR_MOVE_OUT  # 0.2s

# ===================== 融合前：全量Cube → 全量Vector（串行） =====================
def matmul_add_original():
    block_count = 4
    # 步骤1：全量Cube核耗时（4块串行）
    total_cube = block_count * SINGLE_CUBE  # 4×0.4=1.6s
    # 步骤2：全量Vector核耗时（4块串行，等待Cube完成）
    total_vector = block_count * SINGLE_VECTOR  # 4×0.2=0.8s
    # 融合前总耗时 = 全Cube + 全Vector
    return total_cube + total_vector

# ===================== 融合后：按描述的4块流水线并行 =====================
def matmul_add_fused():
    # 步骤1：数据1Cube全流程（无并行）
    step1 = SINGLE_CUBE  # 0.4s

    # 步骤2-4：数据1Vector入+数据2Cube入 → 算 → 出（并行取最大值）
    step2 = max(VECTOR_MOVE_IN, CUBE_MOVE_IN)  # 0.1s
    step3 = max(VECTOR_COMPUTE, CUBE_COMPUTE)  # 0.2s
    step4 = max(VECTOR_MOVE_OUT, CUBE_MOVE_OUT)  # 0.1s

    # 步骤5-7：数据2Vector入+数据3Cube入 → 算 → 出（并行取最大值）
    step5 = max(VECTOR_MOVE_IN, CUBE_MOVE_IN)  # 0.1s
    step6 = max(VECTOR_COMPUTE, CUBE_COMPUTE)  # 0.2s
    step7 = max(VECTOR_MOVE_OUT, CUBE_MOVE_OUT)  # 0.1s

    # 步骤8-10：数据3Vector入+数据4Cube入 → 算 → 出（并行取最大值）
    step8 = max(VECTOR_MOVE_IN, CUBE_MOVE_IN)  # 0.1s
    step9 = max(VECTOR_COMPUTE, CUBE_COMPUTE)  # 0.2s
    step10 = max(VECTOR_MOVE_OUT, CUBE_MOVE_OUT)  # 0.1s

    # 步骤11-13：数据4Vector全流程（无并行）
    step11 = VECTOR_MOVE_IN  # 0.05s
    step12 = VECTOR_COMPUTE  # 0.1s
    step13 = VECTOR_MOVE_OUT  # 0.05s

    # 融合后总耗时 = 所有步骤累加
    total_time = step1 + step2 + step3 + step4 + step5 + step6 + step7 + step8 + step9 + step10 + step11 + step12 + step13
    return total_time

# ===================== 执行对比 =====================
if __name__ == "__main__":
    time_original = matmul_add_original()
    time_fused = matmul_add_fused()
    speedup = (time_original - time_fused) / time_original * 100

    print("=== Ascend C Matmul+Add 融合算子性能演示 ===")
    print(f"✅ 融合前逻辑：全量Cube核Matmul → 全量Vector核Add（串行）")
    print(f"融合前总耗时：{time_original:.2f} 秒（4×0.4 + 4×0.2 = 2.4s）")
    print(f"\n✅ 融合后逻辑：4块数据流水线并行（Cube/Vector核同时工作）")
    print(f"融合后总耗时：{time_fused:.2f} 秒")
    print(f"\n🚀 性能提升：{speedup:.1f}%")


---
## **3. 融合算子的优势**
融合算子是一种优化计算的有效手段，可以提高计算效率和内存利用率，优化数据流，简化代码实现，主要优势如下：  

- 独立的矢量算子和矩阵算子实现：矩阵计算后的结果需要搬运到Global Memory上，然后由Global Memory搬运到Local Memory，再进行矢量算子的计算，计算和搬运都是串行执行，另外多个算子的调度执行，会增加算子的调度耗时。  

- 减少计算量：融合算子可以将多个算子合并为一个，简化计算过程，减少计算量，提高计算效率。  

- 减少内存占用：融合算子可以将多个算子的中间结果合并为一个，从而减少内存占用，提高内存利用率。  

- 优化数据流：融合算子可以优化数据流，减少数据在不同算子之间的传输，从而提高数据处理效率。  

- 简化代码实现：融合算子可以简化代码实现，减少代码量，提高代码可读性和可维护性。  

---

# **课后练习**
请根据本章内容认真思考，选出正确选项。  

1. （单选题）关于Matmul+Add融合算子相比独立算子的核心优势，下列描述错误的是？  
    A. 消除矩阵/矢量算子间的串行数据搬运与调度开销。    
    B. 使Cube和Vector核并行运算，提高硬件利用率。     
    C. 融合后会增加数据在Global/Local Memory间的传输次数。    
    D. 提升算子性能。  


2. （单选题）关于Matmul + Sinh融合算子，下列说法正确的是    
    A. 融合后仅使用Vector核，Matmul和Sinh在核上串行运算。  
    B. 融合后使用Vector和Cube核并行运算。   
    C. 融合后算子运算耗时无变化。  
    D. 融合后由于仍是使用Cube核运算Matmul，Vector核运算Sinh，所以仅减少了代码量，无其他优点。    

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/02.07_answer.txt